In [1]:
import requests
import pandas as pd
from datetime import datetime, timezone

In [2]:
fecha_inicio = datetime(2024, 1, 29, 12, 0, 0, tzinfo=timezone.utc)
fecha_fin = datetime(2024, 1, 29, 13, 0, 0, tzinfo=timezone.utc)

begin = int(fecha_inicio.timestamp())
end = int(fecha_fin.timestamp())

print(begin)
print(end)

1706529600
1706533200


In [3]:
# Guardamos la URL donde OpenSky entrega el token de acceso.
# Esta URL no devuelve vuelos; solo sirve para autenticarnos.
TOKEN_URL = "https://auth.opensky-network.org/auth/realms/opensky-network/protocol/openid-connect/token"

# Guardamos el identificador del cliente API.
# Este valor identifica qué cliente está solicitando acceso.
CLIENT_ID = "ID_client"
# Guardamos el secreto del cliente API.
# Este valor funciona como una contraseña técnica.
# No debe subirse a GitHub con el valor real.
CLIENT_SECRET = "client_secret"
# Enviamos una solicitud POST al servidor de autenticación.
# Usamos POST porque estamos enviando datos al servidor.
token_response = requests.post(
    TOKEN_URL,
    data={
    # Indicamos que usamos el flujo OAuth2 de credenciales de cliente.       
        "grant_type": "client_credentials",
      # Enviamos el identificador del cliente.
        "client_id": CLIENT_ID,
     # Enviamos la clave secreta del cliente.
        "client_secret": CLIENT_SECRET
    }
)
# Mostramos el código de estado HTTP.
# 200 significa que la autenticación fue exitosa.
print(token_response.status_code)
# Mostramos los primeros 300 caracteres de la respuesta.
# Esto permite revisar si llegó un token o un mensaje de error.
print(token_response.text[:...])

401
{"error":"invalid_client","error_description":"Invalid client or Invalid client credentials"}


In [4]:
token = token_response.json()["access_token"]
 #validacion del token 
print(token[:...])

KeyError: 'access_token'

In [ ]:
token = token_response.json()["access_token"]

url = "https://opensky-network.org/api/flights/all"

headers = {
    "Authorization": f"Bearer {token}"
}

params = {
    "begin": begin,
    "end": end
}

response = requests.get(
    url,
    headers=headers,
    params=params
)

print(response.status_code)
print(response.text[:500])

In [ ]:
data = response.json()

print(type(data))
print(data[:2])

In [ ]:
df = pd.DataFrame(data)

df.head()

In [ ]:
df["firstSeen_dt"] = pd.to_datetime(df["firstSeen"], unit="s", utc=True)
df["lastSeen_dt"] = pd.to_datetime(df["lastSeen"], unit="s", utc=True)

df[["firstSeen", "firstSeen_dt", "lastSeen", "lastSeen_dt"]].head()

In [ ]:
df["fecha_salida"] = df["firstSeen_dt"].dt.date
df["hora_salida"] = df["firstSeen_dt"].dt.hour
df["dia_semana"] = df["firstSeen_dt"].dt.day_name()
df["mes"] = df["firstSeen_dt"].dt.month
df["anio"] = df["firstSeen_dt"].dt.year

In [ ]:
df["duracion_minutos"] = (
    df["lastSeen_dt"] - df["firstSeen_dt"]
).dt.total_seconds() / 60

In [ ]:
df[["firstSeen", "lastSeen", "firstSeen_dt", "lastSeen_dt"]].isna().sum()

In [ ]:
df["firstSeen_dt"] = pd.to_datetime(
    df["firstSeen"],
    unit="s",
    utc=True,
    errors="coerce"
)

df["lastSeen_dt"] = pd.to_datetime(
    df["lastSeen"],
    unit="s",
    utc=True,
    errors="coerce"
)

In [ ]:
df["fecha_error"] = df["lastSeen_dt"] < df["firstSeen_dt"]

df[df["fecha_error"] == True]

In [ ]:
columnas_finales = [
    "icao24",
    "callsign",
    "estDepartureAirport",
    "estArrivalAirport",
    "firstSeen_dt",
    "lastSeen_dt",
    "fecha_salida",
    "hora_salida",
    "dia_semana",
    "mes",
    "anio",
    "duracion_minutos"
]

df_final = df[columnas_finales].copy()

In [ ]:
df_final.to_csv(
    "flights_clean_powerbi.csv",
    index=False,
    encoding="utf-8-sig"
)

In [ ]:
import os

print(os.getcwd())